### Telepaartie – Lösung Schritt für Schritt

*38. Bundeswettbewerb Informatik (2019), 1. Runde, Aufgabe 3*

In einem Werk der BWINF AG stehen drei Behälter mit Kuschelbibern. Die Behälter sind so groß, dass zwei Behälter ausreichen würden. Dafür müsste nur umgeschüttet werden. Ohne technische Hilfsmittel geht das aber nicht.

Zum Glück hat die Firmenleitung einen neuen Telepaarteur geschickt. Wenn man ihn auf zwei Behälter richtet und den Auslöser drückt, passiert eine Telepaartie. Sie verändert die Verteilung der Biber in den beiden Behältern so, als würde jeder Biber aus dem weniger vollen Behälter einen Partnerbiber aus dem anderen Behälter zu sich holen: Hat der weniger volle Behälter (bei Gleichstand: einer der beiden Behälter) vorher b Biber, hat er danach b Biber mehr (also 2b Biber), und der andere Behälter hat entsprechend b Biber weniger. Mit wiederholten Telepaartien ist es immer möglich, einen der drei Behälter leer zu bekommen, aber mit wie vielen?

Eine Verteilung von Bibern in den Behältern (eine Biberverteilung) lässt sich durch die Biberanzahlen in den drei Behältern angeben, z. B. (2,4,7). Bei einer gegebenen Verteilung heißt die Anzahl der Telepaartien, die man mindestens braucht, um einen der Behälter leer zu bekommen, die Leerlauflänge (LLL) der Verteilung. Zum Beispiel ist die LLL der Verteilung (2,4,9) genau 2, denn man kann damit anfangen, 2 Biber vom vollsten zum leersten Behälter zu telepaartieren. Es gibt aber auch „schwierigere“ Verteilungen von 15 Bibern, deren LLL größer als 2 ist, zum Beispiel (3,5,7). Für jede Gesamtbiberzahl n sei L(n) die größte LLL von allen Verteilungen von insgesamt n Bibern auf die drei Behälter.

#### Aufgabe

Schreibe ein Programm, das eine Biberverteilung einliest und deren LLL ausgibt. Erweitere das Programm so, dass es eine Zahl n einlesen und die Werte L(1), …, L(n) ausgeben kann. Zeige die Ausgabe für n = 10 und n = 100.

#### Beispieldaten

`beispieldaten/telepaartie.txt`: drei Verteilungen (2, 4, 7), (3, 5, 7) und (80, 64, 32).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zustände und Übergänge:** Eine Verteilung ist ein *Zustand*, eine Telepaartie ein *Übergang* zu einem neuen Zustand. Alle Zustände zusammen mit den Übergängen bilden einen Graphen.
- **Breitensuche (BFS):** Findet in einem Graphen den Weg mit den wenigsten Schritten: erst alle Zustände nach einem Schritt, dann nach zwei Schritten, und so weiter. Der erste Treffer ist garantiert der kürzeste.
- **Normalisieren:** Die Behälter sind austauschbar, (2, 4, 7) und (7, 2, 4) sind für die Aufgabe dasselbe. Wir sortieren jede Verteilung, dann gibt es weniger Zustände.
- **Alle Zustände durchprobieren:** Für L(n) gibt es keine Formel; es gibt aber nur wenige Verteilungen von n Bibern, die wir alle prüfen können.

**Nützliche Funktionen und Bibliotheken**
- `sorted(...)` sortiert drei Zahlen, `tuple(...)` macht daraus ein Tupel, das man in einem Wörterbuch oder einer Menge speichern kann.
- Ein Wörterbuch (`dict`) merkt sich besuchte Zustände samt Vorgänger, `x in woerterbuch` ist schnell.

## Die Idee

**Eine Telepaartie.** Haben zwei Behälter `x ≤ y` Biber, danach haben sie `2x` und `y - x`. Die Summe bleibt gleich. Der dritte Behälter bleibt unverändert. Bei drei Behältern gibt es drei Paare, also höchstens **drei Telepaartien** pro Verteilung.

**Erster Teil (LLL).** Wir suchen den kürzesten Weg von der Startverteilung zu einer Verteilung mit einer 0. Dafür ist die Breitensuche gemacht: Wir halten die Verteilungen fest, die wir nach 0, 1, 2, ... Telepaartien erreichen. Sobald eine dabei ist, in der ein Behälter leer ist, ist die Anzahl der Schritte die LLL. Dass eine 0 immer erreichbar ist, sagt die Aufgabe; die Suche endet also.

**Zweiter Teil (L(n)).** Es gibt nur wenige Verteilungen mit n Bibern (etwa n²/12, wenn man die Reihenfolge der Behälter ignoriert). Wir berechnen die LLL einfach für jede davon und nehmen das Maximum.

## Schritt 1: Einlesen

Die Datei enthält Zeilen wie `Verteilung1: (2, 4, 7)`. Wir nehmen den Teil nach dem Doppelpunkt, entfernen die Klammern und trennen bei den Kommas.

In [ ]:
def lies_verteilungen(dateiname):
    ergebnis = []
    with open(dateiname, encoding='utf-8-sig') as f:
        for zeile in f:
            if ':' in zeile:
                text = zeile.split(':')[1].strip().strip('()')
                zahlen = [int(z) for z in text.split(',')]
                ergebnis.append(tuple(zahlen))
    return ergebnis


VERTEILUNGEN = lies_verteilungen('beispieldaten/telepaartie.txt')
print(VERTEILUNGEN)

[(2, 4, 7), (3, 5, 7), (80, 64, 32)]


## Schritt 2: Eine Telepaartie

`telepaartie(v, i, j)` wendet den Telepaarteur auf die Behälter `i` und `j` der Verteilung `v` an. Der kleinere der beiden Behälter verdoppelt sich, der andere gibt entsprechend ab. Am Ende sortieren wir (Normalisieren).

In [ ]:
def telepaartie(v, i, j):
    """Verteilung v (Tupel aus 3 Zahlen), Telepaartie der Behälter i und j; Ergebnis sortiert."""
    neu = list(v)
    if v[i] <= v[j]:
        neu[i] = 2 * v[i]
        neu[j] = v[j] - v[i]
    else:
        neu[j] = 2 * v[j]
        neu[i] = v[i] - v[j]
    return tuple(sorted(neu))


def nachfolger(v):
    """Alle Verteilungen, die man mit einer Telepaartie aus v erreicht (höchstens 3)."""
    return {telepaartie(v, 0, 1), telepaartie(v, 0, 2), telepaartie(v, 1, 2)}


print(telepaartie((2, 4, 7), 0, 2))     # 2 und 7: der Behälter mit 2 verdoppelt sich, der mit 7 gibt 2 ab
print(telepaartie((2, 4, 9), 0, 2))
print(sorted(nachfolger((2, 4, 7))))

(4, 4, 5)
(4, 4, 7)
[(2, 3, 8), (2, 4, 7), (4, 4, 5)]


Prüfe an den Ausgaben: Bei den Behältern mit 2 und 7 Bibern entstehen 4 und 5, der dritte Behälter behält seine 4 Biber: sortiert `(4, 4, 5)`. Bei Gleichstand (z. B. 4 und 4) wird einer der beiden zu 8, der andere zu 0, das ist dann sofort eine Verteilung mit leerem Behälter.

## Schritt 3: Breitensuche für eine Verteilung

Wir merken uns zu jeder erreichten Verteilung den Vorgänger (in `vorher`), damit wir am Ende den Weg ausgeben können. `ebene` ist die Liste aller Verteilungen, die nach genau `schritte` Telepaartien erstmals erreicht werden.

In [ ]:
def lll(start, mit_weg=False):
    """Leerlauflänge der Verteilung start (und auf Wunsch ein kürzester Weg bis zu einer Verteilung mit leerem Behälter)."""
    start = tuple(sorted(start))
    vorher = {start: None}                   # besuchte Verteilungen mit ihrem Vorgänger
    ebene = [start]
    schritte = 0
    while True:
        for v in ebene:
            if min(v) == 0:                  # ein Behälter ist leer
                if not mit_weg:
                    return schritte
                weg = []
                while v is not None:
                    weg.append(v)
                    v = vorher[v]
                weg.reverse()
                return schritte, weg
        naechste = []
        for v in ebene:
            for w in sorted(nachfolger(v)):
                if w not in vorher:          # noch nie gesehen
                    vorher[w] = v
                    naechste.append(w)
        ebene = naechste
        schritte = schritte + 1


for v in VERTEILUNGEN:
    schritte, weg = lll(v, mit_weg=True)
    print(v, '-> LLL =', schritte)
    print('   Weg:', weg)

(2, 4, 7) -> LLL = 2
   Weg: [(2, 4, 7), (4, 4, 5), (0, 5, 8)]
(3, 5, 7) -> LLL = 3
   Weg: [(3, 5, 7), (2, 6, 7), (4, 4, 7), (0, 7, 8)]
(80, 64, 32) -> LLL = 2
   Weg: [(32, 64, 80), (48, 64, 64), (0, 48, 128)]


Das passt zu den Lösungshinweisen: `(2, 4, 7)` hat LLL 2, `(3, 5, 7)` hat LLL 3 und `(80, 64, 32)` hat LLL 2. Der Weg kann sich vom Weg in den Hinweisen unterscheiden, wenn es mehrere kürzeste Wege gibt; nur die Länge ist eindeutig. Auch die Beispiele aus der Aufgabenstellung stimmen:

In [ ]:
print(lll((2, 4, 9)))
print(lll((3, 5, 7)))
print(lll((7, 4, 2)))        # die Reihenfolge der Behälter spielt keine Rolle

2
3
2


## Schritt 4: Alle Verteilungen von n Bibern

Eine sortierte Verteilung `(a, b, c)` mit `a ≤ b ≤ c` und `a + b + c = n` ist durch `a` und `b` festgelegt. Wir probieren alle solchen `a` und `b` durch und merken uns für jedes `n` den größten Wert der LLL und eine Verteilung dazu.

In [ ]:
def alle_verteilungen(n):
    ergebnis = []
    for a in range(0, n // 3 + 1):
        for b in range(a, (n - a) // 2 + 1):
            c = n - a - b
            ergebnis.append((a, b, c))
    return ergebnis


def groesstes_l(n):
    """L(n) und die erste Verteilung (in der Reihenfolge der Liste), die dieses L(n) erreicht."""
    bestes = -1
    beste_verteilung = None
    for v in alle_verteilungen(n):
        wert = lll(v)
        if wert > bestes:
            bestes = wert
            beste_verteilung = v
    return bestes, beste_verteilung


print(len(alle_verteilungen(15)), 'Verteilungen von 15 Bibern')
print(groesstes_l(15))

27 Verteilungen von 15 Bibern
(4, (2, 5, 8))


Für 15 Biber ist L(15) = 4, wie in den Lösungshinweisen. Dort steht als Beispiel `(8, 4, 3)`, bei uns erscheint die erste gefundene Verteilung mit LLL 4 in unserer Reihenfolge. Es gibt mehrere solche Verteilungen, eindeutig ist nur der Wert L(n).

## Schritt 5: L(1), …, L(n)

In [ ]:
def tabelle(n):
    return [groesstes_l(k)[0] for k in range(1, n + 1)]


L10 = tabelle(10)
for k in range(1, 11):
    print('L(%d) = %d' % (k, L10[k - 1]))

L(1) = 0
L(2) = 0
L(3) = 1
L(4) = 1
L(5) = 1
L(6) = 2
L(7) = 2
L(8) = 2
L(9) = 2
L(10) = 2


Für n = 100 geben wir die Werte in Zehnerzeilen aus. Die Zeile `n=  1..10` zeigt L(1) bis L(10), die Zeile `n= 11..20` zeigt L(11) bis L(20), und so weiter.

In [ ]:
L100 = tabelle(100)
for anfang in range(1, 101, 10):
    werte = L100[anfang - 1:anfang + 9]
    print('n=%3d..%3d:' % (anfang, anfang + 9), ' '.join(str(w) for w in werte))
print('größter Wert bis 100:', max(L100))
beste = groesstes_l(100)
print('L(100) =', beste[0], 'z. B. für', beste[1])

n=  1.. 10: 0 0 1 1 1 2 2 2 2 2
n= 11.. 20: 3 3 3 3 4 3 3 3 4 3
n= 21.. 30: 4 4 5 4 5 4 6 4 5 5
n= 31.. 40: 5 4 6 4 5 4 5 5 6 4
n= 41.. 50: 5 5 6 5 7 5 5 5 6 6
n= 51.. 60: 7 5 6 6 6 5 7 5 6 6
n= 61.. 70: 7 6 7 5 6 7 6 5 7 6
n= 71.. 80: 6 5 6 6 7 6 7 6 6 5
n= 81.. 90: 8 6 7 6 7 6 7 6 7 7
n= 91..100: 7 6 8 6 8 6 7 7 8 7
größter Wert bis 100: 8
L(100) = 7 z. B. für (5, 31, 64)


Die Werte wachsen sehr langsam: Bei bis zu 100 Bibern braucht man höchstens 8 Telepaartien (zum Beispiel bei n = 81, 93, 95 und 99). Sie springen unregelmäßig, es gibt keine einfache Formel (so auch die Lösungshinweise). Zum Vergleich mit den Hinweisen: L(30) = 5, L(40) = 4, L(50) = 6, L(60) = 6, L(70) = 6, L(80) = 5, L(90) = 7 und L(100) = 7, das stimmt mit unseren Werten überein. L(40) ist zum Beispiel kleiner als L(30): Der Wert hängt von n selbst ab, nicht nur von der Größe.

## Für Neugierige

- **Rückwärts von allen Endverteilungen.** Statt für jede Verteilung eine eigene Breitensuche zu starten, kann man alle Distanzen auf einmal berechnen: Alle Verteilungen mit einer 0 haben Distanz 0. Eine Verteilung, von der aus man in einem Schritt eine Verteilung mit Distanz `k - 1` erreicht (und die noch keine Distanz hat), bekommt die Distanz `k`. So geht man Runde für Runde vor. Die Lösungshinweise rechnen genauso (Algorithmus 9, mit vorberechneten Vorgängerlisten); unser Code unten macht es mit einer einfachen Schleife über die Runden.
- **Die Zahl der Zustände** wächst nur quadratisch (etwa n²/12 sortierte Verteilungen), deshalb ist die Rechnung auch für n = 200 schnell.
- **MiniZinc** beschreibt den ersten Teil als Folge von Zuständen (siehe unten).

In [ ]:
def alle_distanzen(n):
    """Distanz (LLL) aller Verteilungen von n Bibern auf einmal."""
    verteilungen = alle_verteilungen(n)
    d = {}
    for v in verteilungen:
        if v[0] == 0:
            d[v] = 0
    runde = 0
    while len(d) < len(verteilungen):
        runde = runde + 1
        neu = []
        for v in verteilungen:
            if v not in d:
                for w in nachfolger(v):
                    if w in d:               # alle bisherigen Einträge haben Distanz kleiner als runde
                        neu.append(v)
                        break
        for v in neu:
            d[v] = runde
    return d


def l_schnell(n):
    return max(alle_distanzen(n).values())


print([l_schnell(k) for k in range(1, 21)])
print(l_schnell(200))

[0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3]
8


Für n = 1 bis 20 kommt dieselbe Folge wie vorher heraus (0 0 1 1 1 2 2 2 2 2 3 3 3 3 4 3 3 3 4 3); L(200) = 8 stimmt mit der Tabelle der Lösungshinweise überein.

## Zum Schluss: Telepaartie mit MiniZinc

Für den **ersten Teil** (LLL einer Verteilung) kann man in MiniZinc einfach die Folge der Zustände beschreiben und die Zahl der Schritte minimieren. Für den zweiten Teil (L(n) als Maximum über alle Verteilungen von dem Minimum über alle Wege) ist MiniZinc nicht das richtige Werkzeug, dort bleibt es bei unserer Rechnung in Python.

**Modell.**

- `x[t, k]` ist die Zahl der Biber in Behälter `k` nach `t` Telepaartien. `x[0, k]` ist die Startverteilung.
- In Schritt `t` wählt der Solver zwei verschiedene Behälter `klein[t]` und `gross[t]`; `klein` hat höchstens so viele Biber wie `gross`. Diese Variablen stehen als **Indizes** in `x`.
- Für die Schritte bis zur benutzten Länge `T` gilt die Telepaartie-Regel, der dritte Behälter bleibt gleich (`->` heißt „wenn, dann“). Danach ändert sich nichts mehr.
- Am Ende (Schritt `H = 8`) ist ein Behälter leer, und wir minimieren `T`. Die Grenze `H = 8` genügt für die Beispiele, die LLL 2 oder 3 haben (und für alle Verteilungen bis 100 Biber, denn dort ist L(n) höchstens 8). Bei größeren Eingaben müsste man `H` vergrößern.

```minizinc
int: H = 8;                                   % so viele Telepaartien höchstens
array[1..3] of int: start;                    % Anfangsverteilung
int: summe = sum(start);

var 0..H: T;                                  % Anzahl der tatsächlich benutzten Telepaartien
array[0..H, 1..3] of var 0..summe: x;         % x[t, k] = Biber in Behälter k nach t Telepaartien
array[1..H] of var 1..3: klein;               % Behälter mit weniger (oder gleich vielen) Bibern
array[1..H] of var 1..3: gross;               % der andere Behälter

constraint forall(k in 1..3)(x[0, k] = start[k]);

constraint forall(t in 1..H)(
  (t <= T ->
     klein[t] != gross[t] /\
     x[t-1, klein[t]] <= x[t-1, gross[t]] /\
     x[t, klein[t]] = 2 * x[t-1, klein[t]] /\
     x[t, gross[t]] = x[t-1, gross[t]] - x[t-1, klein[t]] /\
     forall(k in 1..3)(k != klein[t] /\ k != gross[t] -> x[t, k] = x[t-1, k]))
  /\
  (t > T -> klein[t] = 1 /\ gross[t] = 2 /\ forall(k in 1..3)(x[t, k] = x[t-1, k]))
);

constraint exists(k in 1..3)(x[H, k] = 0);     % am Ende ist ein Behälter leer

solve minimize T;

output ["LLL: ", show(T), "\n", show([[fix(x[t, k]) | k in 1..3] | t in 0..fix(T)])];
```

**Daten** für die erste Verteilung:

```minizinc
start = [2, 4, 7];
```

**Ausgabe:**

```
LLL: 2
[[2, 4, 7], [4, 4, 5], [0, 8, 5]]
----------
==========
```

Ergebnisse für alle drei Beispiele (Solver **Chuffed**, jede Verteilung braucht nur Sekundenbruchteile; Gecode wird dagegen auch nach 15 Sekunden nicht fertig):

```
(2, 4, 7)  ->  LLL 2
(3, 5, 7)  ->  LLL 3
(80, 64, 32)  ->  LLL 2
```

Das sind dieselben Werte wie bei unserer Breitensuche. Anders als die Breitensuche braucht das Modell eine Obergrenze `H` für die Schrittzahl; die Suche selbst weiß von allein, wann sie fertig ist.

In [ ]:
def als_dzn(v):
    return 'start = [' + ', '.join(str(z) for z in v) + '];'


print(als_dzn(VERTEILUNGEN[0]))

start = [2, 4, 7];
